# TalentCLEF TaskA 2025

# Carga de datos

In [2]:
import pandas as pd
import numpy as np

In [3]:
training_data = pd.read_csv("./data/TaskA/training/english/taskA_training_en.tsv", sep='\t', names=['family_id', 'id', 'title1', 'title2'])

validation_data = dict()

validation_data['corpus_elements'] = pd.read_csv("./data/TaskA/validation/english/corpus_elements", sep='\t').set_index('c_id').reset_index(drop=True)

validation_data['queries'] = pd.read_csv("./data/TaskA/validation/english/queries", sep='\t').set_index('q_id').reset_index(drop=True)

validation_data['qrels'] = pd.read_csv("./data/TaskA/validation/english/qrels.tsv", sep='\t', names=['q_id', 'iter', 'c_id', 'relevance'])[['q_id', 'c_id', 'relevance']]
validation_data['qrels']['q_id'] = validation_data['qrels']['q_id'] - 1
validation_data['qrels']['c_id'] = validation_data['qrels']['c_id'] - 1

In [4]:
validation_data['qrels']

,q_id,c_id,relevance
0,0,142,1
1,0,149,1
2,0,763,1
3,0,869,1
4,0,1463,1
...,...,...,...
2415,104,2122,1
2416,104,2143,1
2417,104,2355,1
2418,104,2399,1


In [5]:
print("Training data samples:")
print(training_data[['title1', 'title2']].head())
print("\n")

print("Validation data samples:")
print(validation_data['corpus_elements'].head())
print(validation_data['queries'].head())
print(validation_data['qrels'].head())

Training data samples:
                        title1                       title2
0                air commodore            flight lieutenant
1  command and control officer               flight officer
2                air commodore  command and control officer
3                pilot officer              squadron leader
4       royal airforce officer  command and control officer


Validation data samples:
                           jobtitle
0                recording engineer
1              director of taxation
2  technical support representative
3                        hr manager
4           computer graphic artist
              jobtitle
0                nanny
1    food technologist
2   broadcast engineer
3  automation engineer
4         veterinarian
   q_id  c_id  relevance
0     0   142          1
1     0   149          1
2     0   763          1
3     0   869          1
4     0  1463          1


# Aproximación con embedding ya preentrenado

Se establece device='cpu' debido a que mi ordenador no soporta CUDA

## Configuración

In [6]:
from sentence_transformers import SentenceTransformer, util

# Version multilingue del modelo
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

/home/david/Documents/Master/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def similarity_between_titles(title1, title2):
    emb1 = model.encode(title1, convert_to_tensor=True, device='cpu')
    emb2 = model.encode(title2, convert_to_tensor=True, device='cpu')

    similarity = util.cos_sim(emb1, emb2)

    return similarity.item()

similarity_between_titles("data scientist", "científico de datos")

0.963962197303772

In [8]:
print('Total de casos a revisar:', validation_data['corpus_elements'].shape[0] * validation_data['queries'].shape[0])

Total de casos a revisar: 274995


## Cálculo de similitud

In [ ]:
# calcular la matriz de similitud de query x corpus elements
query_embeddings = model.encode(validation_data['queries']['jobtitle'].tolist(), convert_to_tensor=True, device='cpu', show_progress_bar=True)
corpus_embeddings = model.encode(validation_data['corpus_elements']['jobtitle'].tolist(), convert_to_tensor=True, device='cpu', show_progress_bar=True)

cosine_scores = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()
print('\nMatriz de similitud (query x corpus elements):', cosine_scores.shape)
print(cosine_scores)

Batches: 100%|██████████| 82/82 [00:08<00:00,  9.46it/s]

Matriz de similitud (query x corpus elements): (105, 2619)
[[0.21296816 0.2310408  0.30833417 ... 0.2796646  0.11383708 0.22759824]
 [0.26919237 0.1748625  0.30276865 ... 0.4511308  0.39728913 0.10192341]
 [0.7295592  0.09044953 0.3928468  ... 0.5124594  0.27894348 0.40515035]
 ...
 [0.21932009 0.17697953 0.178989   ... 0.35043252 0.24006447 0.11973904]
 [0.43400556 0.08289627 0.28846532 ... 0.48635602 0.21857396 0.29901707]
 [0.28159153 0.19524384 0.25011295 ... 0.2766195  0.16540116 0.13811007]]


# Busqueda de threshold mediante datos de validación

In [33]:
query_size = validation_data['queries'].shape[0]
corpus_size = validation_data['corpus_elements'].shape[0]

relevance_matrix = np.zeros((query_size, corpus_size), dtype=int)
for index, row in validation_data['qrels'].iterrows():
    relevance_matrix[row['q_id'], row['c_id']] = row['relevance']

relevance_matrix

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(105, 2619))

In [34]:
y_true = relevance_matrix.flatten()
y_scores = cosine_scores.flatten()

from sklearn.metrics import accuracy_score

thresholds = np.linspace(0, 1, 1000)  # 200 valores entre 0 y 1
accuracies = []

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    acc = accuracy_score(y_true, preds)
    accuracies.append(acc)

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Threshold óptimo (accuracy): {best_threshold:.4f}")
print(f"Accuracy: {best_accuracy:.4f}")

Threshold óptimo (accuracy): 0.7598
Accuracy: 0.9925
